In [1]:
import numpy as np
import h5py

import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import librosa
from pathlib import Path
import os

In [17]:
np.__version__

'2.2.2'

In [18]:
df_md = pd.read_csv("audio_metadata_merged_utf-8 - filtered.csv")
df_md_id = np.array(df_md['id']).flatten()

In [19]:
df_md['id']

0        5487
1        5777
2        5768
3        5767
4        5762
        ...  
5305    21047
5306    21040
5307    21049
5308    21050
5309    21039
Name: id, Length: 5310, dtype: int64

In [20]:
df_md_id[:5]

array([5487, 5777, 5768, 5767, 5762])

In [21]:
FOLDER_PATH = "/Users/cheoso/Documents/samples/Packs/"
file_paths = []
total = 0

for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        path = Path(root) / file
        path_lower = str(path).lower()
        if path_lower.endswith(('.wav', '.aif', '.aiff')):
          file_paths.append(str(path))

          total += path.stat().st_size

print(f"Total of {total} bytes.")

Total of 62871673636 bytes.


In [30]:
files = pd.DataFrame(file_paths, columns=['full_file_path'])

In [31]:
files['file_path'] = files['full_file_path'].apply(lambda x: x.lower())
files['file_path'] = files['file_path'].apply(lambda x: x.split('/users/cheoso/documents/samples/packs/')[1])

In [32]:
files.head(5)

,full_file_path,file_path
0,/Users/cheoso/Documents/samples/Packs/Heavywei...,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...
1,/Users/cheoso/Documents/samples/Packs/Heavywei...,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...
2,/Users/cheoso/Documents/samples/Packs/Heavywei...,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...
3,/Users/cheoso/Documents/samples/Packs/Heavywei...,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...
4,/Users/cheoso/Documents/samples/Packs/Heavywei...,heavyweight jungle/eo_hvywt_jngl_one_shots/eo_...


In [10]:
df_md.head(5)

,id,file_path,file_name,group,category,sub_category,Reverse,Forward,One_Shot,Loop,...,Vocal FX,China,Cowbell,Bell,Camera,Orchestra,Metalic Tap,One_Shot_Intent,Loop_id,Loop_Intent
0,5487,808 and kick/heavy kick.wav,Heavy Kick.wav,Drum,Kick,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,0,0,1.0,NaN,0.0
1,5777,808 kingdom - @prodllb/bonus - secrets drum ki...,Clap - Bistro.wav,Percussion,Clap,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,5768,808 kingdom - @prodllb/bonus - secrets drum ki...,Clap - Candy.wav,Percussion,Clap,NaN,0.0,1.0,1.0,0.0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
3,5767,808 kingdom - @prodllb/bonus - secrets drum ki...,Clap - Diddy.wav,Percussion,Clap,NaN,1.0,1.0,1.0,0.0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,5762,808 kingdom - @prodllb/bonus - secrets drum ki...,Clap - Down Wit Me.wav,Percussion,Clap,NaN,1.0,1.0,1.0,0.0,...,0,0,0,0,0,0,0,NaN,NaN,NaN


In [43]:
d = files.merge(df_md[['file_path', 'id']], on='file_path')
d.drop(columns=['file_path'], inplace=True)

In [44]:
d

,full_file_path,id
0,/Users/cheoso/Documents/samples/Packs/Heavywei...,73
1,/Users/cheoso/Documents/samples/Packs/Heavywei...,74
2,/Users/cheoso/Documents/samples/Packs/Heavywei...,75
3,/Users/cheoso/Documents/samples/Packs/Heavywei...,76
4,/Users/cheoso/Documents/samples/Packs/Heavywei...,77
...,...,...
5305,/Users/cheoso/Documents/samples/Packs/Splice C...,38181
5306,/Users/cheoso/Documents/samples/Packs/Splice C...,38184
5307,/Users/cheoso/Documents/samples/Packs/Splice C...,38185
5308,/Users/cheoso/Documents/samples/Packs/Splice C...,38186


In [52]:
d = d.to_numpy()

In [67]:
failed_files = []


def extract_audio_data(index, file):
    """Load an audio file and extract its waveform, sample rate, and label."""
    try:
        file_path = file[0]
        idx = file[1]
        audio, sr = librosa.load(file_path, sr=None)  # Load without resampling
        return [idx, audio, sr]
    except Exception as e:
        failed_files.append(file)
        return None

# Use parallel processing to load audio files
def get_audio(files):
    meta_results = []

    with ThreadPoolExecutor() as executor:
        results = executor.map(lambda p: extract_audio_data(*p), enumerate(files))
        
        for res in results:
            if res is not None:
                meta_data = res
                meta_results.append(meta_data)
    
    return meta_results


audiodataResults = get_audio(d)

In [68]:
audiodataResults = pd.DataFrame(audiodataResults, columns=['id', 'waveform', 'sample_rate'])

In [69]:
audiodataResults.head(5)

,id,waveform,sample_rate
0,73,"[5.9604645e-08, 0.0, 5.9604645e-08, 5.9604645e...",44100
1,74,"[-5.9604645e-08, -5.9604645e-08, -5.9604645e-0...",44100
2,75,"[0.0, -5.9604645e-08, 0.0, 0.0, 5.9604645e-08,...",44100
3,76,"[0.0, 0.0, 0.0, 0.0, 2.3841858e-07, -1.1324883...",44100
4,77,"[5.9604645e-08, 5.9604645e-08, 5.9604645e-08, ...",44100


In [70]:
print(len(audiodataResults))

5310


In [ ]:
diff_sr = audiodataResults[audiodataResults['sample_rate'] != 44100]
diff_sr['waveform'] = diff_sr.apply(lambda x: librosa.resample(x['waveform'], orig_sr=x['sample_rate'], target_sr=44100), axis=1)
diff_sr['sample_rate'] = 44100

/var/folders/h0/pzm07l6x5n7181rc09xppl940000gn/T/ipykernel_70910/1778564183.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diff_sr['waveform'] = diff_sr.apply(lambda x: librosa.resample(x['waveform'], orig_sr=x['sample_rate'], target_sr=44100), axis=1)
/var/folders/h0/pzm07l6x5n7181rc09xppl940000gn/T/ipykernel_70910/1778564183.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diff_sr['sample_rate'] = 44100


In [72]:
audiodataResults.loc[diff_sr.index] = diff_sr

In [73]:
audiodataResults['sample_rate'].min()

np.int64(44100)

In [74]:
audiodataResults['sample_rate'].max()

np.int64(44100)

In [75]:
with h5py.File("test_filtered_audio_data.h5", "w") as f:
    audio_data = f.create_group('audio_data')

    def save_sample(row):
        sample_group = audio_data.create_group(str(row['id']))
        sample_group.create_dataset('waveform', data=row['waveform'], compression='gzip')
        sample_group.create_dataset('sample_rate', data=row['sample_rate'])

    # Use ThreadPoolExecutor for parallelism
    with ThreadPoolExecutor(max_workers=10) as executor:
        list(tqdm(executor.map(save_sample, [row for _, row in audiodataResults.iterrows()]), total=len(audiodataResults)))
    

100%|██████████| 5310/5310 [00:39<00:00, 134.51it/s]
